# 📊 Robust v3 Evaluation — SFT + DPO Adapter Workflow

Self-contained, data-focused evaluation notebook for the v3 training flow:
- **Primary v3 evaluation** — adapter-only load, dynamic token budget, two-phase generation
- **Raw capture diagnostic** — large-token single-phase capture of full outputs
- **Repetition-penalty diagnostic** — compares old anti-loop config against v3 primary inference
- **Phase-aware comparison** — can compare `sft_checkpoint` vs `dpo_checkpoint`

## What this notebook saves
For every model/profile run it saves:
- `evaluation_summary.json`
- `all_results.json`
- `results.csv`
- `evaluation_report.md`
- `raw_outputs/*.txt`
- `parsed_outputs/*.json`
- `think_blocks/*.txt`
- `metadata/*.json`

## Recommended usage
1. Run Cell 1 once, then restart kernel.
2. Edit Cell 2 only.
3. Run through Cell 7.
4. Review summaries in Cell 8 and inspect raw outputs in Cell 9.

This notebook is designed for the exact v3 flow you ran in the training notebook:
**SFT → gate → DPO → adapter-only save → HF push**.

In [ ]:
# ── Cell 1: Install dependencies (run ONCE, then restart kernel) ──────────────
# Same safe install pattern as the training notebook.

import os, glob, shutil, subprocess

USER_SP = os.path.expanduser("~/.local/lib/python3.12/site-packages")

unsloth_dir = os.path.join(USER_SP, "unsloth")
if not os.path.isdir(unsloth_dir):
    print("📦 First run — installing ML packages...")
    subprocess.check_call(
        "pip install unsloth trl datasets transformers accelerate bitsandbytes huggingface_hub peft safetensors sentencepiece protobuf pandas matplotlib seaborn httpx -q".split()
    )
    print("📦 Install complete.")
else:
    print("✅ Packages already installed — skipping pip install")

removed = []
for pattern in ["torch", "torch-*"]:
    for p in glob.glob(os.path.join(USER_SP, pattern)):
        basename = os.path.basename(p)
        if basename.startswith("torchvision") or basename.startswith("torchaudio"):
            continue
        shutil.rmtree(p, ignore_errors=True)
        removed.append(basename)

for p in glob.glob(os.path.join(USER_SP, "nvidia*")):
    shutil.rmtree(p, ignore_errors=True)
    removed.append(os.path.basename(p))

if removed:
    print(f"🗑️  Cleaned from ~/.local: {removed}")
else:
    print("✅ No user-level torch/nvidia to clean")

assert not os.path.exists(os.path.join(USER_SP, "torch")), (
    f"❌ FAILED: torch still in {USER_SP}/torch — please delete manually"
)

r = subprocess.run(
    ["python3", "-c", "import torch; print(torch.__version__, torch.__file__)"],
    capture_output=True,
    text=True,
)
if r.returncode == 0:
    print(f"✅ System torch: {r.stdout.strip()}")
else:
    print(f"⚠️  System torch not found: {r.stderr.strip()[:200]}")

print("\n" + "=" * 60)
print("⚠️  RESTART THE KERNEL (Kernel → Restart)")
print("   Then run Cell 2 onwards. Re-running this cell is safe.")
print("=" * 60)

In [ ]:
# ── Cell 2: CONFIG — edit this cell only ─────────────────────────────────────

CONFIG = {
    # ── Base model for adapter loading ───────────────────────────────────────
    "base_model_hf": "Qwen/Qwen2.5-7B-Instruct",
    "max_seq_length": 4096,
    "load_in_4bit": True,
    "hf_token": "",  # leave empty to use env HF_TOKEN

    # ── Fixed evaluation datasets ─────────────────────────────────────────────
    # IMPORTANT: keep this pointed ONLY at the fixed eval dataset directory.
    # Do NOT add datasets_v2 here, otherwise notebook could silently evaluate on training data.
    "eval_dataset_candidates": [
        "../data/eval_datasets_v2",
        "./data/eval_datasets_v2",
        "./eval_datasets_v2",
    ],
    # Optional training dataset candidates used only for overlap detection.
    "training_dataset_candidates": [
        "../data/datasets_v2",
        "./data/datasets_v2",
        "./datasets_v2",
    ],
    "n_eval": 30,
    "seed": 42,
    "min_support": 3,
    "max_size": 3,
    "require_eval_prefix": True,
    "expected_eval_prefix": "eval_",
    "fail_on_training_overlap": True,

    # ── Models to evaluate ───────────────────────────────────────────────────
    # Toggle enabled=True/False depending on what you want to compare.
    "models": {
        "sft_local": {
            "enabled": True,
            "label": "SFT local adapter",
            "adapter_path": "./sft_checkpoint",
        },
        "dpo_local": {
            "enabled": True,
            "label": "DPO local adapter",
            "adapter_path": "./dpo_checkpoint",
        },
        "dpo_hf": {
            "enabled": False,
            "label": "DPO HuggingFace adapter",
            "adapter_path": "OliverSlivka/qwen2.5-7b-itemset-extractor",
        },
    },

    # ── Evaluation profiles ──────────────────────────────────────────────────
    # primary_v3     = production-style v3 inference
    # raw_capture    = large-budget single-phase capture to inspect real outputs
    # reppenalty     = old anti-loop diagnostic for comparison
    "profiles": {
        "primary_v3": {
            "enabled": True,
            "mode": "two_phase",
            "temperature": 0.3,
            "json_temperature": 0.05,
            "top_k": 50,
            "top_p": 0.90,
            "max_new_tokens_cap": 4096,
            "save_raw": True,
        },
        "raw_capture": {
            "enabled": True,
            "mode": "single_plain",
            "temperature": 0.1,
            "top_k": 50,
            "top_p": 0.95,
            "max_new_tokens": 8192,
            "save_raw": True,
        },
        "reppenalty": {
            "enabled": True,
            "mode": "single_reppenalty",
            "temperature": 0.1,
            "top_k": 50,
            "top_p": 0.95,
            "max_new_tokens": 4096,
            "repetition_penalty": 1.3,
            "no_repeat_ngram_size": 3,
            "save_raw": True,
        },
    },

    # Which profiles should run on which models
    "profile_model_map": {
        "primary_v3": ["sft_local", "dpo_local"],
        "raw_capture": ["dpo_local"],
        "reppenalty": ["dpo_local"],
    },

    # ── Output layout ────────────────────────────────────────────────────────
    "output_root": "./eval_v3_runs",
    "save_parsed_outputs": True,
    "save_think_blocks": True,
    "save_metadata": True,
    "count_tolerance": 1,
}

print("✅ v3 eval CONFIG loaded")
print(f"   Base model: {CONFIG['base_model_hf']}")
print(f"   Max seq length: {CONFIG['max_seq_length']}")
print(f"   N eval: {CONFIG['n_eval']}")
print(f"   Require eval prefix: {CONFIG['require_eval_prefix']} ({CONFIG['expected_eval_prefix']})")
print(f"   Fail on training overlap: {CONFIG['fail_on_training_overlap']}")
print(f"   Profiles enabled: {[k for k, v in CONFIG['profiles'].items() if v['enabled']]}")
print(f"   Models enabled: {[k for k, v in CONFIG['models'].items() if v['enabled']]}")

In [ ]:
# ── Cell 3: GPU check + imports ───────────────────────────────────────────────
import os, sys, json, re, time, gc, random, itertools, warnings, importlib
from pathlib import Path
from datetime import datetime, timezone

os.environ["USE_TF"] = "0"
os.environ["USE_JAX"] = "0"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

warnings.filterwarnings(
    "ignore",
    message=r"The attention mask API under `transformers\.modeling_attn_mask_utils`.*",
    category=FutureWarning,
)

if "torch" in sys.modules:
    raise RuntimeError(
        "Torch is already imported in this kernel before GPU selection. "
        "Restart the kernel, then run from Cell 2 again."
    )


def _query_gpus():
    cmd = [
        "nvidia-smi",
        "--query-gpu=index,name,memory.total,memory.free",
        "--format=csv,noheader,nounits",
    ]
    result = __import__("subprocess").run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"nvidia-smi failed: {result.stderr.strip()[:200]}")

    gpus = []
    for line in result.stdout.strip().splitlines():
        if not line.strip():
            continue
        parts = [part.strip() for part in line.split(",")]
        if len(parts) != 4:
            continue
        gpu_index, name, total_mem, free_mem = parts
        gpus.append({
            "index": int(gpu_index),
            "name": name,
            "total_mem": int(total_mem),
            "free_mem": int(free_mem),
        })
    if not gpus:
        raise RuntimeError("No GPUs found via nvidia-smi")
    return gpus


def _parse_visible_devices(raw_value):
    if not raw_value:
        return None
    raw_value = raw_value.strip()
    if raw_value in {"", "NoDevFiles"}:
        return None
    parsed = []
    for token in raw_value.split(","):
        token = token.strip()
        if not token:
            continue
        if not token.isdigit():
            return None
        parsed.append(int(token))
    return parsed or None


gpus = _query_gpus()
current_visible = _parse_visible_devices(os.environ.get("CUDA_VISIBLE_DEVICES", ""))
available_indexes = {gpu["index"] for gpu in gpus}

if current_visible and all(idx in available_indexes for idx in current_visible):
    selected_gpu = current_visible[0]
    visibility_reason = f"using existing CUDA_VISIBLE_DEVICES={os.environ['CUDA_VISIBLE_DEVICES']}"
else:
    best_gpu = max(gpus, key=lambda gpu: gpu["free_mem"])
    selected_gpu = best_gpu["index"]
    os.environ["CUDA_VISIBLE_DEVICES"] = str(selected_gpu)
    visibility_reason = (
        f"set CUDA_VISIBLE_DEVICES={selected_gpu} "
        f"(auto-selected GPU with most free VRAM)"
    )

print(f"🖥️  GPU visibility: {visibility_reason}")
for gpu in sorted(gpus, key=lambda item: item['index']):
    marker = "← selected" if gpu["index"] == selected_gpu else ""
    print(
        f"   GPU {gpu['index']}: {gpu['name']} | "
        f"free {gpu['free_mem']} MiB / total {gpu['total_mem']} MiB {marker}"
    )

import torch
import pandas as pd
from huggingface_hub import login
from transformers import AutoTokenizer, BitsAndBytesConfig, StoppingCriteria, StoppingCriteriaList
from peft import PeftModel

hf_token = CONFIG["hf_token"] or os.environ.get("HF_TOKEN", "")
if hf_token:
    login(token=hf_token)
    print("✅ HuggingFace logged in")
else:
    print("⚠️  No HF token set — public loading only unless already logged in")

if torch.cuda.is_available():
    torch.cuda.init()
    print(f"\n✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("❌ No GPU found — connect a GPU runtime before continuing.")

In [ ]:
# ── Cell 4: Shared inference + metric utilities ──────────────────────────────
SYSTEM_PROMPT = (
    "You are a frequent itemset extractor. Given CSV transaction data and a "
    "minimum support count, identify all itemsets whose items co-occur in at "
    "least that many rows.\n\n"
    "Rules:\n"
    "1. Scan single items, pairs, and triples (up to size 3)\n"
    "2. Count = number of distinct rows containing ALL items in the itemset\n"
    "3. Only report itemsets with count >= min_support\n"
    "4. Canonicalize items: lowercase, trimmed, sorted alphabetically\n"
    '5. Row references: "Row N" format, 1-based indexing\n\n'
    "Think step by step inside <think> tags, then output ONLY a JSON array:\n"
    '[{"itemset": ["item1", "item2"], "count": N, "rows": ["Row 1", "Row 3"]}]' 
)


class ThinkStoppingCriteria(StoppingCriteria):
    def __init__(self, tokenizer, max_think_tokens: int = 3000):
        super().__init__()
        self.tokenizer = tokenizer
        self.max_think_tokens = max_think_tokens
        self.stop_ids = tokenizer.encode("</think>", add_special_tokens=False)
        self.stop_len = len(self.stop_ids)
        self._generated_count = 0

    def __call__(self, input_ids, scores, **kwargs):
        self._generated_count += 1
        if self._generated_count >= self.max_think_tokens:
            return True
        if input_ids.shape[1] >= self.stop_len:
            last_tokens = input_ids[0, -self.stop_len:].tolist()
            if last_tokens == self.stop_ids:
                return True
        return False


class RepetitionDetector(StoppingCriteria):
    def __init__(self, tokenizer, max_repeats: int = 3, check_every: int = 50):
        super().__init__()
        self.tokenizer = tokenizer
        self.max_repeats = max_repeats
        self.check_every = check_every
        self._step = 0

    def __call__(self, input_ids, scores, **kwargs):
        self._step += 1
        if self._step % self.check_every != 0:
            return False
        text = self.tokenizer.decode(input_ids[0, -500:], skip_special_tokens=True)
        lines = [line.strip() for line in text.split("\n") if line.strip()]
        if len(lines) < self.max_repeats * 2:
            return False
        recent = lines[-self.max_repeats:]
        if len(set(recent)) == 1 and len(recent) >= self.max_repeats:
            return True
        return False


def count_unique_values(csv_text: str) -> int:
    values = set()
    for line in csv_text.strip().split("\n"):
        if ":" in line:
            items_part = line.split(":", 1)[1].strip()
            for item in items_part.split(","):
                item = item.strip().lower()
                if item:
                    values.add(item)
    return len(values)


def calculate_dynamic_budget(n_rows: int, n_cols: int, n_unique_vals: int = 0) -> int:
    if n_unique_vals == 0:
        n_unique_vals = n_cols * 3
    singles_est = n_unique_vals
    pairs_est = min(singles_est * (singles_est - 1) // 4, 50)
    triples_est = min(pairs_est // 3, 20)
    cot_tokens = singles_est * 15 + pairs_est * 20 + triples_est * 25 + 80
    json_tokens = (singles_est + pairs_est + triples_est) * 60
    budget = int((cot_tokens + json_tokens) * 1.5)
    return max(512, min(budget, 6000))


def build_messages(csv_text: str, min_support: int):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Find all frequent itemsets with minimum support count = {min_support} "
                f"in this dataset:\n\n{csv_text}"
            ),
        },
    ]


def prepare_inputs(tokenizer, csv_text: str, min_support: int, device: str = "cuda"):
    messages = build_messages(csv_text, min_support)
    chat_inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    )
    if isinstance(chat_inputs, torch.Tensor):
        input_ids = chat_inputs.to(device)
        attention_mask = torch.ones_like(input_ids, device=input_ids.device)
    else:
        chat_inputs = chat_inputs.to(device)
        input_ids = chat_inputs["input_ids"]
        attention_mask = chat_inputs.get("attention_mask")
        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids, device=input_ids.device)
    return input_ids, attention_mask


def generate_two_phase(
    model,
    tokenizer,
    input_ids,
    attention_mask=None,
    think_temperature: float = 0.3,
    json_temperature: float = 0.05,
    think_max_tokens: int = 2000,
    json_max_tokens: int = 1500,
    top_k: int = 50,
    top_p: float = 0.90,
    think_do_sample: bool = True,
    json_do_sample: bool = True,
):
    think_stopper = ThinkStoppingCriteria(tokenizer, max_think_tokens=think_max_tokens)
    rep_detector = RepetitionDetector(tokenizer, max_repeats=3)

    phase1_kwargs = {
        "input_ids": input_ids,
        "max_new_tokens": think_max_tokens,
        "do_sample": think_do_sample,
        "pad_token_id": tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
        "stopping_criteria": StoppingCriteriaList([think_stopper, rep_detector]),
    }
    if attention_mask is not None:
        phase1_kwargs["attention_mask"] = attention_mask
    if think_do_sample:
        phase1_kwargs.update({
            "temperature": think_temperature,
            "top_k": top_k,
            "top_p": top_p,
        })

    with torch.no_grad():
        think_output = model.generate(**phase1_kwargs)

    think_text = tokenizer.decode(think_output[0][input_ids.shape[1]:], skip_special_tokens=True)
    if "</think>" in think_text:
        think_text = think_text.split("</think>")[0] + "</think>\n"
    else:
        lines_local = think_text.split("\n")
        last_good = len(lines_local) - 1
        for i in range(len(lines_local) - 1, -1, -1):
            if any(marker in lines_local[i] for marker in ["✓", "✗", "##", "RESULT", "FREQUENT", "SCAN"]):
                last_good = i
                break
        think_text = "\n".join(lines_local[:last_good + 1]) + "\n</think>\n"

    prompt_text = tokenizer.decode(input_ids[0], skip_special_tokens=False)
    json_prompt = prompt_text + think_text + "["
    json_inputs = tokenizer(json_prompt, return_tensors="pt").to(input_ids.device)

    phase2_kwargs = {
        "input_ids": json_inputs["input_ids"],
        "attention_mask": json_inputs["attention_mask"],
        "max_new_tokens": json_max_tokens,
        "do_sample": json_do_sample,
        "pad_token_id": tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }
    if json_do_sample:
        phase2_kwargs.update({
            "temperature": json_temperature,
            "top_k": 20,
            "top_p": 0.95,
        })

    with torch.no_grad():
        json_output = model.generate(**phase2_kwargs)

    json_text = tokenizer.decode(
        json_output[0][json_inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

    full_response = think_text
    if not full_response.rstrip().endswith("</think>"):
        full_response = full_response.rstrip() + "\n</think>\n"
    full_response += "[" + json_text
    return full_response


def generate_single_plain(model, tokenizer, input_ids, attention_mask, profile_cfg):
    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=profile_cfg["max_new_tokens"],
            temperature=profile_cfg["temperature"],
            top_k=profile_cfg.get("top_k", 50),
            top_p=profile_cfg.get("top_p", 0.95),
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True)


def generate_single_reppenalty(model, tokenizer, input_ids, attention_mask, profile_cfg):
    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=profile_cfg["max_new_tokens"],
            temperature=profile_cfg["temperature"],
            top_k=profile_cfg.get("top_k", 50),
            top_p=profile_cfg.get("top_p", 0.95),
            do_sample=True,
            repetition_penalty=profile_cfg["repetition_penalty"],
            no_repeat_ngram_size=profile_cfg["no_repeat_ngram_size"],
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True)


def extract_and_repair_json(raw_output: str):
    has_think = "<think>" in raw_output and "</think>" in raw_output
    json_text = raw_output
    if has_think:
        last_think_idx = raw_output.rfind("</think>")
        json_text = raw_output[last_think_idx + len("</think>"):].strip()

    try:
        parsed = json.loads(json_text)
        if isinstance(parsed, list):
            return parsed, True, json_text
    except json.JSONDecodeError:
        pass

    match = re.search(r"\[.*?\]", json_text, re.DOTALL)
    if match:
        try:
            parsed = json.loads(match.group())
            if isinstance(parsed, list):
                return parsed, True, match.group()
        except json.JSONDecodeError:
            pass

    match = re.search(r"\[.*\]", json_text, re.DOTALL)
    if match:
        try:
            parsed = json.loads(match.group())
            if isinstance(parsed, list):
                return parsed, True, match.group()
        except json.JSONDecodeError:
            pass

    return [], False, json_text


def load_transactions_csv(csv_path: Path):
    df = pd.read_csv(csv_path)
    n_rows, n_cols = df.shape
    transactions, rows_text, all_items = [], [], set()
    for idx, row in df.iterrows():
        items = [str(v).strip() for v in row.values if pd.notna(v) and str(v).strip()]
        transactions.append(items)
        rows_text.append(f"Row {idx + 1}: {', '.join(items)}")
        for item in items:
            all_items.add(str(item).strip().lower())
    return {
        "dataset": csv_path.name,
        "csv_path": str(csv_path),
        "csv_text": "\n".join(rows_text),
        "transactions": transactions,
        "n_rows": n_rows,
        "n_cols": n_cols,
        "all_items": all_items,
    }


def apriori_frequent_itemsets(transactions, min_support=3, max_size=3):
    if not transactions:
        return []
    row_labels = [f"Row {i + 1}" for i in range(len(transactions))]
    counts = {}
    for idx, trans in enumerate(transactions):
        seen = set()
        for item in trans:
            norm = str(item).strip().lower()
            if not norm or norm in seen:
                continue
            seen.add(norm)
            key = (norm,)
            counts.setdefault(key, {"count": 0, "rows": []})
            counts[key]["count"] += 1
            counts[key]["rows"].append(row_labels[idx])

    def prune(level):
        return {k: v for k, v in level.items() if v["count"] >= min_support}

    current = prune(counts)
    if not current:
        return []
    levels = [current]
    k = 2
    while k <= max_size and current:
        prev_keys = sorted(current.keys())
        candidates = set()
        for i in range(len(prev_keys)):
            for j in range(i + 1, len(prev_keys)):
                a, b = prev_keys[i], prev_keys[j]
                if a[: k - 2] == b[: k - 2]:
                    merged = tuple(sorted(set(a) | set(b)))
                    if len(merged) == k:
                        if all(tuple(sorted(sub)) in current for sub in itertools.combinations(merged, k - 1)):
                            candidates.add(merged)
        if not candidates:
            break
        cand_counts = {cand: {"count": 0, "rows": []} for cand in candidates}
        for idx, trans in enumerate(transactions):
            tset = {str(x).strip().lower() for x in trans}
            for cand in candidates:
                if set(cand).issubset(tset):
                    cand_counts[cand]["count"] += 1
                    cand_counts[cand]["rows"].append(row_labels[idx])
        current = prune(cand_counts)
        if current:
            levels.append(current)
        k += 1

    output = []
    for level in levels:
        for itemset, info in level.items():
            output.append({
                "itemset": list(itemset),
                "count": info["count"],
                "rows": info["rows"],
            })
    return output


def canon(itemset):
    return frozenset(str(x).strip().lower() for x in itemset)


def extract_hash_suffix(path: Path):
    stem_parts = path.stem.split("_")
    if not stem_parts:
        return None
    candidate = stem_parts[-1].lower()
    return candidate if re.fullmatch(r"[0-9a-f]{8,12}", candidate) else None


def find_existing_directory(candidates):
    paths = [Path(p) for p in candidates]
    return next((path for path in paths if path.exists() and path.is_dir()), None)


def collect_training_hashes():
    training_dir = find_existing_directory(CONFIG.get("training_dataset_candidates", []))
    if training_dir is None:
        return None, set()
    hashes = set()
    for csv_path in training_dir.glob("*.csv"):
        hash_suffix = extract_hash_suffix(csv_path)
        if hash_suffix:
            hashes.add(hash_suffix)
    return training_dir, hashes


def validate_eval_dataset_selection(selected_dir: Path, csv_files):
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in {selected_dir}")

    names = [path.name for path in csv_files]
    prefix = CONFIG.get("expected_eval_prefix", "eval_")
    if CONFIG.get("require_eval_prefix", False):
        bad_names = [name for name in names if not name.startswith(prefix)]
        if bad_names:
            preview = bad_names[:5]
            raise RuntimeError(
                "Selected dataset directory does not look like the fixed eval set. "
                f"Expected filenames starting with '{prefix}', got examples: {preview}. "
                f"Directory used: {selected_dir}"
            )

    training_dir, training_hashes = collect_training_hashes()
    selected_hashes = {extract_hash_suffix(path) for path in csv_files if extract_hash_suffix(path)}
    overlap = sorted(hash_value for hash_value in selected_hashes if hash_value in training_hashes)

    validation = {
        "selected_dir": str(selected_dir),
        "selected_count": len(csv_files),
        "training_dir": str(training_dir) if training_dir else None,
        "training_hash_count": len(training_hashes),
        "selected_hash_count": len(selected_hashes),
        "hash_overlap_count": len(overlap),
        "hash_overlap": overlap,
        "prefix_required": bool(CONFIG.get("require_eval_prefix", False)),
        "expected_prefix": prefix,
        "prefix_ok": all(name.startswith(prefix) for name in names),
    }

    if CONFIG.get("fail_on_training_overlap", False) and overlap:
        raise RuntimeError(
            "Eval dataset overlap detected with training datasets. "
            f"Overlapping hashes: {overlap[:10]} | selected_dir={selected_dir} | training_dir={training_dir}"
        )

    return validation


def normalize_model_items(parsed):
    items = []
    for rec in parsed:
        if not isinstance(rec, dict):
            continue
        itemset = rec.get("itemset", [])
        if not isinstance(itemset, list) or not itemset:
            continue
        count = rec.get("count", 0)
        rows = rec.get("rows", rec.get("row", rec.get("evidence_rows", [])))
        norm_itemset = sorted(str(x).strip().lower() for x in itemset)
        norm_rows = []
        if not isinstance(rows, list):
            rows = []
        for row in rows:
            if isinstance(row, int):
                norm_rows.append(f"Row {row}")
            elif isinstance(row, str) and re.match(r"Row \d+", row):
                norm_rows.append(row)
            else:
                try:
                    norm_rows.append(f"Row {int(row)}")
                except (TypeError, ValueError):
                    pass
        items.append({
            "itemset": norm_itemset,
            "count": count if isinstance(count, int) else 0,
            "rows": sorted(set(norm_rows)),
        })
    return items


def compute_metrics(apriori_sets, model_sets, all_csv_items, count_tolerance=1):
    apr_canonical = {canon(s["itemset"]) for s in apriori_sets}
    mod_canonical = {canon(s["itemset"]) for s in model_sets}

    tp = apr_canonical & mod_canonical
    fp = mod_canonical - apr_canonical
    fn = apr_canonical - mod_canonical

    precision = len(tp) / len(mod_canonical) if mod_canonical else 0.0
    recall = len(tp) / len(apr_canonical) if apr_canonical else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    apr_map = {canon(s["itemset"]): s for s in apriori_sets}
    count_correct = 0
    row_correct = 0
    count_total = 0
    row_total = 0
    for pred in model_sets:
        key = canon(pred["itemset"])
        if key in apr_map:
            count_total += 1
            row_total += 1
            if abs(pred["count"] - apr_map[key]["count"]) <= count_tolerance:
                count_correct += 1
            if sorted(pred["rows"]) == sorted(apr_map[key]["rows"]):
                row_correct += 1

    count_accuracy = count_correct / count_total if count_total > 0 else 0.0
    row_accuracy = row_correct / row_total if row_total > 0 else 0.0

    hallucinated = 0
    for pred in model_sets:
        if any(item not in all_csv_items for item in pred["itemset"]):
            hallucinated += 1
    hallucination_rate = hallucinated / len(model_sets) if model_sets else 0.0

    strict_exact_match = not fp and not fn and count_accuracy == 1.0 and row_accuracy == 1.0

    return {
        "apriori_count": len(apr_canonical),
        "model_count": len(mod_canonical),
        "tp": len(tp),
        "fp": len(fp),
        "fn": len(fn),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "exact_match": f1 == 1.0,
        "strict_exact_match": strict_exact_match,
        "count_accuracy": round(count_accuracy, 4),
        "row_accuracy": round(row_accuracy, 4),
        "hallucination_rate": round(hallucination_rate, 4),
    }


def detect_failure_patterns(raw_output, model_sets, all_csv_items, parse_ok, hit_limit):
    lines_local = [line.strip() for line in raw_output.split("\n") if line.strip()]
    tail_loop = len(lines_local) >= 3 and len(set(lines_local[-3:])) == 1
    itemset_keys = [tuple(item["itemset"]) for item in model_sets]
    duplicate_itemsets = len(itemset_keys) != len(set(itemset_keys))
    contains_colval = any(":" in token for item in model_sets for token in item["itemset"])
    hallucinated_itemsets = sum(
        1 for item in model_sets if any(token not in all_csv_items for token in item["itemset"])
);
    return {
        "parse_ok": bool(parse_ok),
        "has_think": "<think>" in raw_output,
        "has_closed_think": "</think>" in raw_output,
        "contains_colval": contains_colval,
        "hit_limit": bool(hit_limit),
        "tail_loop": bool(tail_loop),
        "duplicate_itemsets": bool(duplicate_itemsets),
        "hallucinated_itemsets": hallucinated_itemsets,
    }


def build_eval_records():
    candidate_dirs = [Path(p) for p in CONFIG["eval_dataset_candidates"]]
    existing_dir = next((path for path in candidate_dirs if path.exists() and path.is_dir()), None)
    if existing_dir is None:
        raise FileNotFoundError(
            f"No eval dataset directory found. Checked: {[str(p) for p in candidate_dirs]}"
        )
    csv_files = sorted(existing_dir.glob("*.csv"))
    selection_validation = validate_eval_dataset_selection(existing_dir, csv_files)

    random.seed(CONFIG["seed"])
    if len(csv_files) > CONFIG["n_eval"]:
        selected = random.sample(csv_files, CONFIG["n_eval"])
        selected = sorted(selected)
    else:
        selected = csv_files

    records = []
    for csv_path in selected:
        record = load_transactions_csv(csv_path)
        record["ground_truth"] = apriori_frequent_itemsets(
            record["transactions"],
            min_support=CONFIG["min_support"],
            max_size=CONFIG["max_size"],
        )
        records.append(record)
    return existing_dir, records, selection_validation


def build_markdown_report(summary, results):
    lines_local = [
        "# v3 Evaluation Report",
        "",
        f"**Model key:** {summary['model_key']}",
        f"**Model label:** {summary['model_label']}",
        f"**Profile:** {summary['profile_key']}",
        f"**Datasets:** {summary['successful']}/{summary['total']}",
        f"**Timestamp:** {summary['timestamp']}",
        "",
        "## Aggregate Metrics",
        "",
        "| Metric | Value |",
        "|--------|-------|",
        f"| Avg F1 | {summary['avg_f1']:.1%} |",
        f"| Avg Precision | {summary['avg_precision']:.1%} |",
        f"| Avg Recall | {summary['avg_recall']:.1%} |",
        f"| Exact Match | {summary['exact_match_rate']:.1%} |",
        f"| Strict Exact Match | {summary['strict_exact_match_rate']:.1%} |",
        f"| JSON Parse Rate | {summary['parse_rate']:.1%} |",
        f"| Think Rate | {summary['think_rate']:.1%} |",
        f"| Closed Think Rate | {summary['closed_think_rate']:.1%} |",
        f"| Col:Val Rate | {summary['colval_rate']:.1%} |",
        f"| Hallucination Rate | {summary['avg_hallucination']:.1%} |",
        f"| Count Accuracy | {summary['avg_count_accuracy']:.1%} |",
        f"| Row Accuracy | {summary['avg_row_accuracy']:.1%} |",
        f"| Hit Limit Rate | {summary['hit_limit_rate']:.1%} |",
        f"| Tail Loop Rate | {summary['tail_loop_rate']:.1%} |",
        f"| Avg Time | {summary['avg_time_s']:.1f}s |",
        "",
        "## Per-Dataset Results",
        "",
        "| Dataset | Rows×Cols | P | R | F1 | Parse | Think | ColVal | Halluc | Time |",
        "|---------|-----------|---|---|----|-------|-------|--------|--------|------|",
    ]
    for result in results:
        if "metrics" not in result:
            continue
        metrics = result["metrics"]
        patterns = result["patterns"]
        lines_local.append(
            f"| {result['dataset']} | {result['n_rows']}×{result['n_cols']} | "
            f"{metrics['precision']:.0%} | {metrics['recall']:.0%} | {metrics['f1']:.0%} | "
            f"{'✅' if patterns['parse_ok'] else '❌'} | {'✅' if patterns['has_think'] else '❌'} | "
            f"{'✅' if patterns['contains_colval'] else '❌'} | {metrics['hallucination_rate']:.0%} | {result['model_time_s']:.1f}s |"
        )
    return "\n".join(lines_local)


print("✅ Shared helper utilities ready")
print("   • Adapter-only HF-native loading")
print("   • Primary v3 two-phase inference")
print("   • Raw capture + reppenalty diagnostics")
print("   • Eval dataset validation: prefix + training-overlap checks")
print("   • Official-style metrics + extra failure pattern tracking")

In [ ]:
# ── Cell 5: Load fixed evaluation datasets ───────────────────────────────────
eval_dataset_dir, eval_records, eval_selection_validation = build_eval_records()

print(f"📂 Using eval datasets from: {eval_dataset_dir}")
print(f"✅ Loaded {len(eval_records)} evaluation records")
print("✅ Eval dataset validation:")
print(f"   Prefix OK: {eval_selection_validation['prefix_ok']}")
print(f"   Expected prefix: {eval_selection_validation['expected_prefix']}")
print(f"   Training dir checked: {eval_selection_validation['training_dir']}")
print(f"   Training hash count: {eval_selection_validation['training_hash_count']}")
print(f"   Selected hash count: {eval_selection_validation['selected_hash_count']}")
print(f"   Hash overlap count: {eval_selection_validation['hash_overlap_count']}")
print("   Sample datasets:")
for record in eval_records[:5]:
    print(
        f"   - {record['dataset']} | {record['n_rows']} rows × {record['n_cols']} cols | "
        f"Apriori={len(record['ground_truth'])} itemsets"
    )

In [ ]:
# ── Cell 6: Model loading + evaluation engine ────────────────────────────────
CURRENT_MODEL = None
CURRENT_TOKENIZER = None
CURRENT_MODEL_KEY = None


def clear_loaded_model():
    global CURRENT_MODEL, CURRENT_TOKENIZER, CURRENT_MODEL_KEY
    if CURRENT_MODEL is not None:
        del CURRENT_MODEL
    if CURRENT_TOKENIZER is not None:
        del CURRENT_TOKENIZER
    CURRENT_MODEL = None
    CURRENT_TOKENIZER = None
    CURRENT_MODEL_KEY = None
    gc.collect()
    torch.cuda.empty_cache()


def load_adapter_model(model_key, model_cfg):
    global CURRENT_MODEL, CURRENT_TOKENIZER, CURRENT_MODEL_KEY
    if CURRENT_MODEL is not None and CURRENT_MODEL_KEY == model_key:
        return CURRENT_MODEL, CURRENT_TOKENIZER

    clear_loaded_model()

    import transformers.models.qwen2.modeling_qwen2 as _qwen2_mod
    _qwen2_mod = importlib.reload(_qwen2_mod)
    _Qwen2ForCausalLM = _qwen2_mod.Qwen2ForCausalLM

    import peft.peft_model as _peft_mod
    _peft_mod = importlib.reload(_peft_mod)
    _PeftModel = _peft_mod.PeftModel

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=CONFIG["load_in_4bit"],
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    print(f"\n📥 Loading {model_key}: {model_cfg['label']}")
    base_model = _Qwen2ForCausalLM.from_pretrained(
        CONFIG["base_model_hf"],
        quantization_config=bnb_config,
        attn_implementation="sdpa",
        dtype=torch.bfloat16,
        device_map={"": 0},
    )

    tokenizer = AutoTokenizer.from_pretrained(CONFIG["base_model_hf"])
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    if tokenizer.bos_token_id is None and getattr(base_model.config, "bos_token_id", None) is not None:
        tokenizer.bos_token_id = base_model.config.bos_token_id

    model = _PeftModel.from_pretrained(base_model, model_cfg["adapter_path"])
    model.eval()
    if hasattr(model, "generation_config") and hasattr(model.generation_config, "max_length"):
        model.generation_config.max_length = None

    CURRENT_MODEL = model
    CURRENT_TOKENIZER = tokenizer
    CURRENT_MODEL_KEY = model_key

    print(f"✅ Loaded adapter on HF-native SDPA base")
    print(f"   VRAM allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    return CURRENT_MODEL, CURRENT_TOKENIZER


def run_profile_inference(model, tokenizer, record, profile_cfg):
    input_ids, attention_mask = prepare_inputs(tokenizer, record["csv_text"], CONFIG["min_support"])
    prompt_tokens = int(input_ids.shape[1])
    started = time.perf_counter()

    if profile_cfg["mode"] == "two_phase":
        n_unique = max(1, len(record["all_items"]))
        total_budget = min(
            profile_cfg["max_new_tokens_cap"],
            calculate_dynamic_budget(record["n_rows"], record["n_cols"], n_unique),
        )
        think_budget = max(768, min(2000, int(total_budget * 0.55)))
        json_budget = max(512, total_budget - think_budget)
        raw_output = generate_two_phase(
            model,
            tokenizer,
            input_ids,
            attention_mask=attention_mask,
            think_temperature=profile_cfg["temperature"],
            json_temperature=profile_cfg["json_temperature"],
            think_max_tokens=think_budget,
            json_max_tokens=json_budget,
            top_k=profile_cfg["top_k"],
            top_p=profile_cfg["top_p"],
        )
        budget_meta = {
            "mode": "two_phase",
            "total_budget": total_budget,
            "think_budget": think_budget,
            "json_budget": json_budget,
        }
        hit_limit = False
    elif profile_cfg["mode"] == "single_plain":
        raw_output = generate_single_plain(model, tokenizer, input_ids, attention_mask, profile_cfg)
        budget_meta = {
            "mode": "single_plain",
            "total_budget": profile_cfg["max_new_tokens"],
        }
        output_tokens = len(tokenizer(raw_output, add_special_tokens=False)["input_ids"])
        hit_limit = output_tokens >= profile_cfg["max_new_tokens"] - 10
    elif profile_cfg["mode"] == "single_reppenalty":
        raw_output = generate_single_reppenalty(model, tokenizer, input_ids, attention_mask, profile_cfg)
        budget_meta = {
            "mode": "single_reppenalty",
            "total_budget": profile_cfg["max_new_tokens"],
            "repetition_penalty": profile_cfg["repetition_penalty"],
            "no_repeat_ngram_size": profile_cfg["no_repeat_ngram_size"],
        }
        output_tokens = len(tokenizer(raw_output, add_special_tokens=False)["input_ids"])
        hit_limit = output_tokens >= profile_cfg["max_new_tokens"] - 10
    else:
        raise ValueError(f"Unsupported profile mode: {profile_cfg['mode']}")

    elapsed = round(time.perf_counter() - started, 2)
    parsed_raw, parse_ok, parsed_json_text = extract_and_repair_json(raw_output)
    model_sets = normalize_model_items(parsed_raw)
    output_tokens = len(tokenizer(raw_output, add_special_tokens=False)["input_ids"])

    return {
        "raw_output": raw_output,
        "parsed_json_text": parsed_json_text,
        "parse_ok": parse_ok,
        "model_sets": model_sets,
        "prompt_tokens": prompt_tokens,
        "output_tokens": output_tokens,
        "hit_limit": hit_limit,
        "model_time_s": elapsed,
        "budget_meta": budget_meta,
    }


def save_dataset_artifacts(base_dir: Path, result: dict):
    stem = result["dataset"].replace(".csv", "")
    raw_dir = base_dir / "raw_outputs"
    parsed_dir = base_dir / "parsed_outputs"
    parsed_text_dir = base_dir / "parsed_json_text"
    think_dir = base_dir / "think_blocks"
    meta_dir = base_dir / "metadata"
    apriori_dir = base_dir / "apriori_outputs"
    input_dir = base_dir / "csv_inputs"
    raw_dir.mkdir(parents=True, exist_ok=True)
    parsed_dir.mkdir(parents=True, exist_ok=True)
    parsed_text_dir.mkdir(parents=True, exist_ok=True)
    think_dir.mkdir(parents=True, exist_ok=True)
    meta_dir.mkdir(parents=True, exist_ok=True)
    apriori_dir.mkdir(parents=True, exist_ok=True)
    input_dir.mkdir(parents=True, exist_ok=True)

    (raw_dir / f"{stem}_raw.txt").write_text(result["raw_output"], encoding="utf-8")
    (parsed_dir / f"{stem}_parsed_normalized.json").write_text(
        json.dumps(result["model_output"], indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    parsed_text_path = parsed_text_dir / f"{stem}_parsed_fragment.txt"
    parsed_text_path.write_text(result.get("parsed_json_text", ""), encoding="utf-8")
    if result.get("parsed_json_text", "").strip():
        try:
            parsed_obj = json.loads(result["parsed_json_text"])
            (parsed_text_dir / f"{stem}_parsed_fragment.json").write_text(
                json.dumps(parsed_obj, indent=2, ensure_ascii=False),
                encoding="utf-8",
            )
        except json.JSONDecodeError:
            pass
    think_text = result["raw_output"].split("</think>", 1)[0] if "</think>" in result["raw_output"] else ""
    (think_dir / f"{stem}_think.txt").write_text(think_text, encoding="utf-8")
    (apriori_dir / f"{stem}_apriori.json").write_text(
        json.dumps(result["apriori_output"], indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    (input_dir / f"{stem}_input.txt").write_text(result.get("csv_text", ""), encoding="utf-8")
    meta_payload = {
        key: value
        for key, value in result.items()
        if key not in {"raw_output", "apriori_output", "model_output", "csv_text"}
    }
    (meta_dir / f"{stem}_meta.json").write_text(
        json.dumps(meta_payload, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )


def build_artifact_inventory(run_dir: Path):
    files = []
    for path in sorted(run_dir.rglob("*")):
        if path.is_file():
            rel = path.relative_to(run_dir)
            files.append({
                "path": str(rel),
                "parent": str(rel.parent),
                "suffix": path.suffix,
                "size_bytes": path.stat().st_size,
            })
    return files


def aggregate_results(model_key, model_cfg, profile_key, results):
    successful = [result for result in results if "metrics" in result]
    n_total = len(results)
    n_success = len(successful)
    if not successful:
        return {
            "model_key": model_key,
            "model_label": model_cfg["label"],
            "profile_key": profile_key,
            "total": n_total,
            "successful": 0,
            "failed": n_total,
            "timestamp": datetime.now(timezone.utc).isoformat(),
        }

    summary = {
        "model_key": model_key,
        "model_label": model_cfg["label"],
        "profile_key": profile_key,
        "total": n_total,
        "successful": n_success,
        "failed": n_total - n_success,
        "avg_precision": sum(r["metrics"]["precision"] for r in successful) / n_success,
        "avg_recall": sum(r["metrics"]["recall"] for r in successful) / n_success,
        "avg_f1": sum(r["metrics"]["f1"] for r in successful) / n_success,
        "exact_match_rate": sum(1 for r in successful if r["metrics"]["exact_match"]) / n_success,
        "strict_exact_match_rate": sum(1 for r in successful if r["metrics"]["strict_exact_match"]) / n_success,
        "parse_rate": sum(1 for r in successful if r["patterns"]["parse_ok"]) / n_success,
        "think_rate": sum(1 for r in successful if r["patterns"]["has_think"]) / n_success,
        "closed_think_rate": sum(1 for r in successful if r["patterns"]["has_closed_think"]) / n_success,
        "colval_rate": sum(1 for r in successful if r["patterns"]["contains_colval"]) / n_success,
        "avg_hallucination": sum(r["metrics"]["hallucination_rate"] for r in successful) / n_success,
        "avg_count_accuracy": sum(r["metrics"]["count_accuracy"] for r in successful) / n_success,
        "avg_row_accuracy": sum(r["metrics"]["row_accuracy"] for r in successful) / n_success,
        "hit_limit_rate": sum(1 for r in successful if r["patterns"]["hit_limit"]) / n_success,
        "tail_loop_rate": sum(1 for r in successful if r["patterns"]["tail_loop"]) / n_success,
        "duplicate_itemset_rate": sum(1 for r in successful if r["patterns"]["duplicate_itemsets"]) / n_success,
        "avg_time_s": sum(r["model_time_s"] for r in successful) / n_success,
        "avg_prompt_tokens": sum(r["prompt_tokens"] for r in successful) / n_success,
        "avg_output_tokens": sum(r["output_tokens"] for r in successful) / n_success,
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }
    for key, value in list(summary.items()):
        if isinstance(value, float):
            summary[key] = round(value, 4)
    return summary


def evaluate_profile_model(model_key, model_cfg, profile_key, profile_cfg, records, run_dir: Path):
    model, tokenizer = load_adapter_model(model_key, model_cfg)
    result_dir = run_dir / profile_key / model_key
    result_dir.mkdir(parents=True, exist_ok=True)
    results = []

    print("\n" + "=" * 80)
    print(f"🚀 Evaluating model={model_key} | profile={profile_key}")
    print("=" * 80)

    for index, record in enumerate(records, start=1):
        print(f"[{index}/{len(records)}] {record['dataset']} ({record['n_rows']}x{record['n_cols']})")
        inference = run_profile_inference(model, tokenizer, record, profile_cfg)
        metrics = compute_metrics(
            record["ground_truth"],
            inference["model_sets"],
            record["all_items"],
            count_tolerance=CONFIG["count_tolerance"],
        )
        patterns = detect_failure_patterns(
            inference["raw_output"],
            inference["model_sets"],
            record["all_items"],
            inference["parse_ok"],
            inference["hit_limit"],
        )
        result = {
            "dataset": record["dataset"],
            "csv_path": record["csv_path"],
            "csv_text": record["csv_text"],
            "n_rows": record["n_rows"],
            "n_cols": record["n_cols"],
            "apriori_count": len(record["ground_truth"]),
            "model_count": len({canon(item['itemset']) for item in inference['model_sets']}),
            "prompt_tokens": inference["prompt_tokens"],
            "output_tokens": inference["output_tokens"],
            "model_time_s": inference["model_time_s"],
            "budget_meta": inference["budget_meta"],
            "parsed_json_text": inference["parsed_json_text"],
            "metrics": metrics,
            "patterns": patterns,
            "raw_output": inference["raw_output"],
            "apriori_output": record["ground_truth"],
            "model_output": inference['model_sets'],
        }
        results.append(result)
        save_dataset_artifacts(result_dir, result)
        print(
            f"   → P={metrics['precision']:.0%} R={metrics['recall']:.0%} F1={metrics['f1']:.0%} | "
            f"JSON={'✅' if patterns['parse_ok'] else '❌'} | Think={'✅' if patterns['has_think'] else '❌'} | "
            f"ColVal={'✅' if patterns['contains_colval'] else '❌'} | Limit={'⚠️' if patterns['hit_limit'] else '✅'}"
        )

    summary = aggregate_results(model_key, model_cfg, profile_key, results)
    (result_dir / "evaluation_summary.json").write_text(
        json.dumps(summary, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    serializable_results = []
    for result in results:
        serializable_results.append({
            key: value for key, value in result.items() if key != "raw_output"
        })
    (result_dir / "all_results.json").write_text(
        json.dumps(serializable_results, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    pd.DataFrame([
        {
            "dataset": result["dataset"],
            "n_rows": result["n_rows"],
            "n_cols": result["n_cols"],
            "precision": result["metrics"]["precision"],
            "recall": result["metrics"]["recall"],
            "f1": result["metrics"]["f1"],
            "exact_match": result["metrics"]["exact_match"],
            "strict_exact_match": result["metrics"]["strict_exact_match"],
            "count_accuracy": result["metrics"]["count_accuracy"],
            "row_accuracy": result["metrics"]["row_accuracy"],
            "hallucination_rate": result["metrics"]["hallucination_rate"],
            "parse_ok": result["patterns"]["parse_ok"],
            "has_think": result["patterns"]["has_think"],
            "has_closed_think": result["patterns"]["has_closed_think"],
            "contains_colval": result["patterns"]["contains_colval"],
            "hit_limit": result["patterns"]["hit_limit"],
            "tail_loop": result["patterns"]["tail_loop"],
            "duplicate_itemsets": result["patterns"]["duplicate_itemsets"],
            "prompt_tokens": result["prompt_tokens"],
            "output_tokens": result["output_tokens"],
            "model_time_s": result["model_time_s"],
        }
        for result in results
    ]).to_csv(result_dir / "results.csv", index=False)
    (result_dir / "evaluation_report.md").write_text(
        build_markdown_report(summary, results),
        encoding="utf-8",
    )
    return summary, results, result_dir


print("✅ Evaluation engine ready")
print("   • Per dataset now saves raw, normalized parsed, parsed fragment, think, apriori, metadata, and input text")

In [ ]:
# ── Cell 7: Run full v3 evaluation suite ─────────────────────────────────────
enabled_models = {k: v for k, v in CONFIG["models"].items() if v["enabled"]}
enabled_profiles = {k: v for k, v in CONFIG["profiles"].items() if v["enabled"]}

if not enabled_models:
    raise RuntimeError("No models enabled in CONFIG['models']")
if not enabled_profiles:
    raise RuntimeError("No profiles enabled in CONFIG['profiles']")

run_stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
run_dir = Path(CONFIG["output_root"]) / f"run_{run_stamp}"
run_dir.mkdir(parents=True, exist_ok=True)

all_summaries = []
all_result_dirs = []

for profile_key, profile_cfg in enabled_profiles.items():
    target_model_keys = CONFIG["profile_model_map"].get(profile_key, [])
    target_model_keys = [key for key in target_model_keys if key in enabled_models]
    if not target_model_keys:
        print(f"⚠️  Skipping profile {profile_key} — no enabled models mapped")
        continue

    for model_key in target_model_keys:
        model_cfg = enabled_models[model_key]
        summary, results, result_dir = evaluate_profile_model(
            model_key,
            model_cfg,
            profile_key,
            profile_cfg,
            eval_records,
            run_dir,
        )
        all_summaries.append(summary)
        all_result_dirs.append(str(result_dir))

clear_loaded_model()

summary_df = pd.DataFrame(all_summaries).sort_values(
    by=["avg_f1", "parse_rate", "colval_rate"],
    ascending=[False, False, False],
)
summary_df.to_csv(run_dir / "cross_profile_summary.csv", index=False)
(run_dir / "cross_profile_summary.json").write_text(
    json.dumps(all_summaries, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
manifest = {
    "run_dir": str(run_dir),
    "eval_dataset_dir": str(eval_dataset_dir),
    "eval_dataset_validation": eval_selection_validation,
    "n_eval": len(eval_records),
    "profiles": list(enabled_profiles.keys()),
    "models": list(enabled_models.keys()),
    "result_dirs": all_result_dirs,
    "timestamp": datetime.now(timezone.utc).isoformat(),
}
(run_dir / "run_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

artifact_inventory = build_artifact_inventory(run_dir)
(run_dir / "artifact_inventory.json").write_text(
    json.dumps(artifact_inventory, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
artifact_inventory_df = pd.DataFrame(artifact_inventory)
artifact_inventory_df.to_csv(run_dir / "artifact_inventory.csv", index=False)

print("\n✅ Full evaluation suite complete")
print(f"   Run dir: {run_dir}")
print(f"   Saved files: {len(artifact_inventory_df)}")
print(f"   Eval dir validated: {eval_selection_validation['selected_dir']}")
print(f"   Prefix OK: {eval_selection_validation['prefix_ok']}")
print(f"   Training overlap count: {eval_selection_validation['hash_overlap_count']}")
print("   Summary table:")
display(summary_df)

In [ ]:
# ── Cell 8: Highlight best runs + pass/fail verdict ──────────────────────────
if summary_df.empty:
    raise RuntimeError("No evaluation summaries found. Run Cell 7 first.")

best_by_f1 = summary_df.sort_values("avg_f1", ascending=False).iloc[0]
best_by_parse = summary_df.sort_values("parse_rate", ascending=False).iloc[0]
best_by_halluc = summary_df.sort_values("avg_hallucination", ascending=True).iloc[0]

print("=" * 80)
print("🏁 v3 EVALUATION HIGHLIGHTS")
print("=" * 80)
print("Best by F1:")
print(best_by_f1[["model_key", "profile_key", "avg_f1", "avg_precision", "avg_recall", "parse_rate", "colval_rate", "avg_hallucination"]])
print("\nBest by Parse Rate:")
print(best_by_parse[["model_key", "profile_key", "parse_rate", "avg_f1", "colval_rate", "hit_limit_rate", "tail_loop_rate"]])
print("\nBest by Lowest Hallucination:")
print(best_by_halluc[["model_key", "profile_key", "avg_hallucination", "avg_f1", "parse_rate", "count_accuracy", "avg_time_s"] if "count_accuracy" in best_by_halluc.index else ["model_key", "profile_key", "avg_hallucination", "avg_f1", "parse_rate", "avg_time_s"]])

production_like = summary_df[summary_df["profile_key"] == "primary_v3"].copy()
if not production_like.empty:
    print("\nPrimary v3 profile verdicts:")
    for _, row in production_like.iterrows():
        reasons = []
        if row["avg_f1"] < 0.80:
            reasons.append(f"F1 {row['avg_f1']:.1%} < 80%")
        if row["parse_rate"] < 0.90:
            reasons.append(f"Parse {row['parse_rate']:.1%} < 90%")
        if row["avg_hallucination"] > 0.05:
            reasons.append(f"Hallucination {row['avg_hallucination']:.1%} > 5%")
        verdict = "🎉 PASS" if not reasons else f"⚠️ NOT YET — {', '.join(reasons)}"
        print(f"   {row['model_key']}: {verdict}")

In [ ]:
# ── Cell 9: Show all saved artifacts for this run ─────────────────────────────
if 'run_dir' not in globals():
    raise RuntimeError("Run Cell 7 first so run_dir exists.")

inventory_path = Path(run_dir) / "artifact_inventory.csv"
if not inventory_path.exists():
    raise FileNotFoundError(f"Missing artifact inventory: {inventory_path}")

artifact_inventory_df = pd.read_csv(inventory_path)
print(f"📁 Run dir: {run_dir}")
print(f"📄 Total saved files: {len(artifact_inventory_df)}")
print("\nFiles by folder:")
display(
    artifact_inventory_df.groupby("parent", dropna=False)["path"]
    .count()
    .reset_index(name="files")
    .sort_values(["files", "parent"], ascending=[False, True])
)

print("\nFirst 200 saved files:")
display(artifact_inventory_df.head(200))

In [ ]:
# ── Cell 10: Inspect one dataset's saved inputs and outputs ───────────────────
# Edit these values after Cell 7 if you want to inspect a specific artifact.
INSPECT_PROFILE = "primary_v3"
INSPECT_MODEL = "dpo_local"
MANUAL_DATASET_STEM = ""

if MANUAL_DATASET_STEM:
    artifact_dir = Path(run_dir) / INSPECT_PROFILE / INSPECT_MODEL
    raw_path = artifact_dir / "raw_outputs" / f"{MANUAL_DATASET_STEM}_raw.txt"
    parsed_path = artifact_dir / "parsed_outputs" / f"{MANUAL_DATASET_STEM}_parsed_normalized.json"
    parsed_fragment_path = artifact_dir / "parsed_json_text" / f"{MANUAL_DATASET_STEM}_parsed_fragment.txt"
    apriori_path = artifact_dir / "apriori_outputs" / f"{MANUAL_DATASET_STEM}_apriori.json"
    input_path = artifact_dir / "csv_inputs" / f"{MANUAL_DATASET_STEM}_input.txt"
    meta_path = artifact_dir / "metadata" / f"{MANUAL_DATASET_STEM}_meta.json"
    print(f"Input path: {input_path}")
    print(f"Raw path: {raw_path}")
    print(f"Parsed normalized path: {parsed_path}")
    print(f"Parsed fragment path: {parsed_fragment_path}")
    print(f"Apriori path: {apriori_path}")
    print(f"Meta path: {meta_path}")
    if input_path.exists():
        print("\n─── INPUT CSV TEXT ───")
        print(input_path.read_text(encoding="utf-8")[:2500])
    if raw_path.exists():
        raw_text = raw_path.read_text(encoding="utf-8")
        print("\n─── RAW HEAD ───")
        print(raw_text[:2500])
        print("\n─── RAW TAIL ───")
        print(raw_text[-2500:])
    if parsed_fragment_path.exists():
        print("\n─── PARSED JSON FRAGMENT ───")
        print(parsed_fragment_path.read_text(encoding="utf-8")[:2500])
    if parsed_path.exists():
        print("\n─── PARSED NORMALIZED JSON ───")
        print(parsed_path.read_text(encoding="utf-8")[:2500])
    if apriori_path.exists():
        print("\n─── APRIORI GROUND TRUTH ───")
        print(apriori_path.read_text(encoding="utf-8")[:2500])
    if meta_path.exists():
        print("\n─── METADATA ───")
        print(meta_path.read_text(encoding="utf-8")[:2500])
else:
    print("Set MANUAL_DATASET_STEM to something like 'eval_001_15x9_51a8fe492fc7' and rerun this cell.")
    print(f"Run dir available at: {run_dir}")